# Hive + Spark + PySpark Integration Demo

This notebook demonstrates:
1. Connecting PySpark to Hive Metastore
2. Creating Hive tables from Spark
3. Reading/writing data through Hive
4. Running Spark SQL queries on Hive tables
5. Loading data from HDFS into Hive tables

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc
import pyspark.sql.types as T

# ── Spark Session with Hive support ──
spark = SparkSession.builder \
    .appName("HiveSparkDemo") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.hive.metastore.version", "4.1.0") \
    .config("spark.sql.hive.metastore.jars", "/opt/hive/lib/*") \
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9083") \
    .enableHiveSupport() \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Hive catalog enabled: {spark.conf.get('spark.sql.catalogImplementation')}")

Spark version: 4.1.2
Hive catalog enabled: hive


## 1. Explore Hive Metastore

In [ ]:
# List all databases in Hive
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|demo_db  |
+---------+



In [ ]:
# Create a new database
spark.sql("CREATE DATABASE IF NOT EXISTS demo_db")
spark.sql("USE demo_db")
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|demo_db  |
+---------+



## 2. Create Hive Tables from PySpark

In [ ]:
# Create a managed Hive table using Spark SQL
spark.sql("""
    CREATE TABLE IF NOT EXISTS employees (
        id      INT,
        name    STRING,
        role    STRING,
        salary  DOUBLE,
        dept    STRING
    )
    STORED AS PARQUET
    TBLPROPERTIES ('comment'='Employee records')
""")
print("✅ Table 'employees' created")

# Show table metadata from Hive
spark.sql("DESCRIBE EXTENDED employees").select("col_name", "data_type", "comment").show(10, truncate=False)

✅ Table 'employees' created
+----------------------------+-------------+-------+
|col_name                    |data_type    |comment|
+----------------------------+-------------+-------+
|id                          |int          |NULL   |
|name                        |string       |NULL   |
|role                        |string       |NULL   |
|salary                      |double       |NULL   |
|dept                        |string       |NULL   |
|                            |             |       |
|# Detailed Table Information|             |       |
|Catalog                     |spark_catalog|       |
|Database                    |demo_db      |       |
|Table                       |employees    |       |
+----------------------------+-------------+-------+
only showing top 10 rows


In [ ]:
# Create an external table pointing to HDFS data
spark.sql("""
    CREATE TABLE IF NOT EXISTS sales_external (
        sale_id    INT,
        product    STRING,
        amount     DOUBLE,
        sale_date  DATE
    )
    STORED AS PARQUET
    LOCATION '/user/hive/warehouse/sales_data'
""")
print("✅ External table 'sales_external' created")

✅ External table 'sales_external' created


## 3. Insert Data

In [ ]:
# Insert rows using Spark SQL
spark.sql("""
    INSERT INTO employees VALUES
        (1, 'Alice',   'Engineer',   95000, 'Engineering'),
        (2, 'Bob',     'Analyst',    75000, 'Analytics'),
        (3, 'Charlie', 'Manager',   120000, 'Engineering'),
        (4, 'Diana',   'Analyst',    82000, 'Analytics'),
        (5, 'Eve',     'Engineer',  105000, 'Engineering'),
        (6, 'Frank',   'Director',  150000, 'Executive'),
        (7, 'Grace',   'Engineer',   98000, 'Engineering'),
        (8, 'Hank',    'Analyst',    72000, 'Analytics')
""")
print("✅ Data inserted into 'employees'")

In [ ]:
# Insert using a DataFrame
from datetime import date

sales_data = [
    (101, 'Widget',   250.00, date(2026, 6, 1)),
    (102, 'Gadget',   499.99, date(2026, 6, 2)),
    (103, 'Widget',   300.00, date(2026, 6, 3)),
    (104, 'Doohickey', 89.99, date(2026, 6, 4)),
    (105, 'Gadget',   550.00, date(2026, 6, 5)),
]
df_sales = spark.createDataFrame(sales_data, schema=["sale_id", "product", "amount", "sale_date"])
df_sales.write.insertInto("sales_external")
print("✅ Data inserted into 'sales_external'")

## 4. Query with Spark SQL

In [ ]:
# Basic query
spark.sql("""
    SELECT dept, COUNT(*) as count, ROUND(AVG(salary), 0) as avg_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""").show()

[Stage 3:>                                                          (0 + 8) / 8]

+-----------+-----+----------+
|       dept|count|avg_salary|
+-----------+-----+----------+
|  Executive|    1|  150000.0|
|Engineering|    4|  104500.0|
|  Analytics|    3|   76333.0|
+-----------+-----+----------+



In [ ]:
# Window functions
spark.sql("""
    SELECT name, dept, salary,
           RANK() OVER (PARTITION BY dept ORDER BY salary DESC) as dept_rank
    FROM employees
    ORDER BY dept, dept_rank
""").show()

+-------+-----------+--------+---------+
|   name|       dept|  salary|dept_rank|
+-------+-----------+--------+---------+
|  Diana|  Analytics| 82000.0|        1|
|    Bob|  Analytics| 75000.0|        2|
|   Hank|  Analytics| 72000.0|        3|
|Charlie|Engineering|120000.0|        1|
|    Eve|Engineering|105000.0|        2|
|  Grace|Engineering| 98000.0|        3|
|  Alice|Engineering| 95000.0|        4|
|  Frank|  Executive|150000.0|        1|
+-------+-----------+--------+---------+



In [ ]:
# CTE + multi-join
spark.sql("""
    WITH dept_stats AS (
        SELECT dept,
               COUNT(*) as headcount,
               ROUND(AVG(salary), 0) as avg_sal
        FROM employees
        GROUP BY dept
    )
    SELECT e.name, e.dept, e.salary,
           d.avg_sal,
           ROUND(e.salary - d.avg_sal, 0) as diff_from_avg
    FROM employees e
    JOIN dept_stats d ON e.dept = d.dept
    ORDER BY diff_from_avg DESC
""").show()

## 5. DataFrame API on Hive Tables

In [ ]:
# Load Hive table as DataFrame
df = spark.table("employees")
df.printSchema()
df.show(5)

In [ ]:
# DataFrame transformations
from pyspark.sql.functions import when

df \
    .withColumn("level",
        when(col("salary") >= 120000, "Senior")
        .when(col("salary") >= 90000, "Mid")
        .otherwise("Junior")
    ) \
    .groupBy("level") \
    .agg(count("*").alias("count"), avg("salary").alias("avg_salary")) \
    .orderBy(desc("avg_salary")) \
    .show()

NameError: name 'df' is not defined

## 6. Join Hive Tables

In [ ]:
# Cross-table query
spark.sql("""
    SELECT e.name, e.dept, s.product, s.amount, s.sale_date
    FROM employees e
    CROSS JOIN sales_external s
    WHERE e.dept = 'Analytics'
    ORDER BY s.sale_date, e.name
    LIMIT 20
""").show()

+-----+---------+---------+------+----------+
| name|     dept|  product|amount| sale_date|
+-----+---------+---------+------+----------+
|  Bob|Analytics|   Widget| 250.0|2026-06-01|
|Diana|Analytics|   Widget| 250.0|2026-06-01|
| Hank|Analytics|   Widget| 250.0|2026-06-01|
|  Bob|Analytics|   Gadget|499.99|2026-06-02|
|Diana|Analytics|   Gadget|499.99|2026-06-02|
| Hank|Analytics|   Gadget|499.99|2026-06-02|
|  Bob|Analytics|   Widget| 300.0|2026-06-03|
|Diana|Analytics|   Widget| 300.0|2026-06-03|
| Hank|Analytics|   Widget| 300.0|2026-06-03|
|  Bob|Analytics|Doohickey| 89.99|2026-06-04|
|Diana|Analytics|Doohickey| 89.99|2026-06-04|
| Hank|Analytics|Doohickey| 89.99|2026-06-04|
|  Bob|Analytics|   Gadget| 550.0|2026-06-05|
|Diana|Analytics|   Gadget| 550.0|2026-06-05|
| Hank|Analytics|   Gadget| 550.0|2026-06-05|
+-----+---------+---------+------+----------+



## 7. Write Results to Hive Table

In [ ]:
# Save query result as a new Hive table
spark.sql("""
    CREATE TABLE IF NOT EXISTS dept_summary
    STORED AS PARQUET
    AS
    SELECT dept,
           COUNT(*)    as headcount,
           ROUND(AVG(salary), 0) as avg_salary,
           ROUND(MAX(salary), 0) as max_salary,
           ROUND(MIN(salary), 0) as min_salary
    FROM employees
    GROUP BY dept
""")
print("✅ Table 'dept_summary' created from query")
spark.table("dept_summary").show()

## 6. Load CSV — from Local filesystem and from HDFS

In [ ]:
# ── Load CSV from the LOCAL filesystem into a Hive table ──
import os, tempfile, subprocess
local_csv = os.path.join(tempfile.gettempdir(), "nb_people.csv")
with open(local_csv, "w") as f:
    f.write("name,age,city\nAlice,30,NYC\nBob,25,LA\nCarol,35,SF\nDan,28,NYC\n")

df_local = (spark.read.option("header", True).option("inferSchema", True)
            .csv("file://" + local_csv))
print("CSV read from LOCAL filesystem:")
df_local.show()
df_local.write.mode("overwrite").saveAsTable("people_from_local")
print("✅ saved to Hive table people_from_local:", spark.table("people_from_local").count(), "rows")

In [ ]:
# ── Upload the CSV to HDFS, then read it back from hadoop into a Hive table ──
hdfs_dir = "/user/%s/nb_csv" % os.environ.get("USER", "hadoop")
hdfs_csv = hdfs_dir + "/people.csv"
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", hdfs_dir])
subprocess.run(["hdfs", "dfs", "-put", "-f", local_csv, hdfs_csv])
print("Uploaded to HDFS:")
subprocess.run(["hdfs", "dfs", "-ls", hdfs_dir])

df_hdfs = (spark.read.option("header", True).option("inferSchema", True)
           .csv("hdfs://localhost:9000" + hdfs_csv))
print("CSV read from HDFS:")
df_hdfs.show()
df_hdfs.write.mode("overwrite").saveAsTable("people_from_hdfs")
spark.sql("SELECT city, COUNT(*) c, ROUND(AVG(age),1) avg_age "
          "FROM people_from_hdfs GROUP BY city ORDER BY c DESC").show()
print("✅ local rows == hdfs rows:", df_local.count() == df_hdfs.count())

## 8. Verify Everything is in Hive Metastore

All tables should be visible from the Hive CLI too:
```bash
hive -e "USE demo_db; SHOW TABLES;"
```

In [ ]:
spark.sql("USE demo_db")
spark.sql("SHOW TABLES").show(truncate=False)

In [ ]:
# ── Cleanup (optional) ──
# Uncomment to drop demo tables
# spark.sql("DROP TABLE IF EXISTS dept_summary")
# spark.sql("DROP TABLE IF EXISTS employees")
# spark.sql("DROP TABLE IF EXISTS sales_external")
# spark.sql("DROP DATABASE IF EXISTS demo_db CASCADE")
# print("🧹 Cleanup done")

In [ ]:
spark.stop()